# Step 09: Deploy RF-DETR Model to Azure Kubernetes Service (AKS)
### Production Microservice Deployment with Internal LoadBalancer (Zero Public IP)
Directly packages the fine-tuned RF-DETR ONNX runtime model into an enterprise Kubernetes microservice.

### Architectural Guarantees:
- **Zero Public IP**: Internal LoadBalancer with private IP ingress.
- **Triton / FastAPI Ingress**: Preprocessing + ONNX inference.
- **Automated Kubernetes Rollout**: `deployment.yaml` and `service.yaml` with health probes.


In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q azure-cli kubernetes requests pillow

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Cloud Infrastructure Configuration (Zero Hardcoded Secrets)
ACR_NAME = os.getenv("ACR_NAME", "myacrregistry")
AKS_CLUSTER = os.getenv("AKS_CLUSTER_NAME", "myakscluster")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "myresourcegroup")
NAMESPACE = os.getenv("AKS_NAMESPACE", "rfdetr-serving")
IMAGE_TAG = f"{ACR_NAME}.azurecr.io/rfdetr-multi-class:v1"

logger.info(f"Target ACR:        {ACR_NAME}")
logger.info(f"Target AKS:        {AKS_CLUSTER} (RG: {RESOURCE_GROUP})")
logger.info(f"Target Namespace:  {NAMESPACE}")
logger.info(f"Image Tag:         {IMAGE_TAG}")

In [ ]:
# CELL 3 - Generate Kubernetes Manifests
k8s_dir = Path("deployment") / "k8s"
k8s_dir.mkdir(parents=True, exist_ok=True)

manifest_yaml = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: rfdetr-multi-class-service
  namespace: {NAMESPACE}
spec:
  replicas: 2
  selector:
    matchLabels:
      app: rfdetr-multi-class
  template:
    metadata:
      labels:
        app: rfdetr-multi-class
    spec:
      containers:
      - name: rfdetr-container
        image: {IMAGE_TAG}
        resources:
          requests:
            cpu: "2"
            memory: "4Gi"
          limits:
            cpu: "4"
            memory: "8Gi"
        ports:
        - containerPort: 8000
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 15
          periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: rfdetr-internal-service
  namespace: {NAMESPACE}
  annotations:
    service.beta.kubernetes.io/azure-load-balancer-internal: "true"
spec:
  type: LoadBalancer
  ports:
  - port: 80
    targetPort: 8000
  selector:
    app: rfdetr-multi-class
"""

with open(k8s_dir / "service_deployment.yaml", "w") as f:
    f.write(manifest_yaml)

logger.info(f"✓ Generated Kubernetes manifests in: {k8s_dir / 'service_deployment.yaml'}")

In [ ]:
# CELL 4 - Health Check & Smoke Test Client
def test_local_service(endpoint_url: str):
    logger.info(f"Checking endpoint: {endpoint_url}/health...")
    try:
        resp = requests.get(f"{endpoint_url}/health", timeout=5)
        logger.info(f"Service status: {resp.status_code} - {resp.json()}")
    except Exception as e:
        logger.info(f"Endpoint not reachable yet (expected prior to cluster rollout): {e}")

test_local_service("http://127.0.0.1:8000")